# Single-Player Puzzle MCTS + LLM Optimization Loop

This notebook demonstrates the two-component optimization loop on **single-player puzzles**:

1. **Component 1 — MCTS plays puzzles** with pluggable heuristic functions
2. **Component 2 — LLM analyzes traces** and proposes improved heuristics

**Architecture modules used** (all inside `mcts/`):

| Module | Responsibility |
|---|---|
| `MCTSEngine` | Core MCTS with 4 pluggable heuristic slots |
| `LLMClient` | Ollama REST API wrapper (handles thinking-mode) |
| `PromptBuilder` | Game-specific prompt templates + assembly |
| `HeuristicLoader` | Code extraction, `exec()`, validation |
| `OptimizationLoop` | Full play→analyse→improve orchestrator |

The key insight: MCTS with *random* rollouts rarely solves puzzles.
An LLM can discover domain-specific heuristics (e.g., Manhattan distance)
that dramatically improve solve rates.

## Part 1: Understanding the Puzzle Games

In [9]:
# Cell 1: Imports — all from the mcts package
from mcts import (
    MCTSEngine, Game, GameState,
    LLMClient, PromptBuilder, HeuristicLoader,
    OptimizationLoop,
)
from mcts.games.sliding_puzzle import SlidingPuzzle, SlidingPuzzleState
from mcts.games.sokoban import Sokoban, SokobanState

print("Imports OK")
print(f"Available games: SlidingPuzzle, Sokoban")

Imports OK
Available games: SlidingPuzzle, Sokoban


In [10]:
# Cell 2: Explore the Sliding Puzzle
game = SlidingPuzzle(size=3, scramble_moves=10, max_steps=50)
state = game.new_initial_state()

print(f'Game: {game.name()}')
print(f'Num players: {game.num_players()}')
print(f'State type: {type(state).__name__}')
print()
print('Initial (scrambled) puzzle:')
print(state)
print()
print(f'Legal actions: {state.legal_actions()}')
print(f'Action names: UP=0, DOWN=1, LEFT=2, RIGHT=3  (direction blank moves)')
print(f'Misplaced tiles: {state.misplaced_tiles()}')
print(f'Manhattan distance: {state.manhattan_distance()}')
print(f'Is terminal: {state.is_terminal()}')
print(f'Player: {state.current_player()} (always 0 for single-player)')

Game: Sliding Puzzle 3x3
Num players: 1
State type: SlidingPuzzleState

Initial (scrambled) puzzle:
Step 0/50 | Misplaced: 7 | Manhattan: 10
+--+--+--+
|4 |1 |3 |
+--+--+--+
|7 |  |2 |
+--+--+--+
|8 |6 |5 |
+--+--+--+

Legal actions: [0, 1, 2, 3]
Action names: UP=0, DOWN=1, LEFT=2, RIGHT=3  (direction blank moves)
Misplaced tiles: 7
Manhattan distance: 10
Is terminal: False
Player: 0 (always 0 for single-player)


In [11]:
# Cell 3: Explore Sokoban
from mcts.games.sokoban import LEVELS

print('Available Sokoban levels:')
for name, lines in LEVELS.items():
    game_s = Sokoban(level_name=name)
    s = game_s.new_initial_state()
    print(f'  {name}: {s.num_targets} boxes, '
          f'{len(s.legal_actions())} initial actions')
print()

# Show micro3 (2 boxes, moderate difficulty)
game_s = Sokoban(level_name='micro3', max_steps=100)
state_s = game_s.new_initial_state()
print(f'\nGame: {game_s.name()}')
print(state_s)

Available Sokoban levels:
  micro1: 1 boxes, 4 initial actions
  micro2: 1 boxes, 3 initial actions
  micro3: 2 boxes, 3 initial actions
  micro4: 2 boxes, 4 initial actions
  micro5: 3 boxes, 3 initial actions


Game: Sokoban (micro3)
Step 0/100 | Boxes on target: 0/2 | Total distance: 2
  ####  
###  ###
# $ .  #
# .$ @ #
###  ###
  ####  


## Part 2: Component 1 — MCTS Baseline (No Heuristics)

Run MCTS with **random rollouts** (default heuristics) on the Sliding Puzzle.
With 10 scramble moves, baseline solve rate is ~20-50%.

In [12]:
# Cell 4: Current heuristics (the LLM optimization targets)
engine = MCTSEngine(
    SlidingPuzzle(size=3, scramble_moves=10, max_steps=50),
    iterations=30,
    max_rollout_depth=50,
)

print('=== Current heuristic functions (LLM targets) ===')
for name, src in engine.get_heuristic_source().items():
    print(f'\n--- {name} ---')
    print(src[:300])

=== Current heuristic functions (LLM targets) ===

--- rollout_policy ---
def random_rollout_policy(state: GameState) -> Any:
    """
    Choose an action during the simulation (rollout) phase.

    Args:
        state: Current game state (non-terminal, has legal actions).

    Returns:
        One of state.legal_actions().

    Default: uniform random.
    The LLM can re

--- evaluation ---
def null_evaluation(state: GameState, perspective_player: int) -> float | None:
    """
    Optionally evaluate a non-terminal state and return a value estimate.

    Args:
        state:              Current (possibly non-terminal) game state.
        perspective_player: The player whose perspectiv

--- exploration_weight ---
def default_exploration_weight(root_visits: int) -> float:
    """
    Return the exploration constant C for UCB1.

    Args:
        root_visits: How many total simulations the root has run so far.

    Returns:
        A positive float (typically ~1.4).

    The LLM can imp

In [13]:
# Cell 5: Play baseline games
NUM_GAMES = 30
SCRAMBLE = 10
ITERS = 30

game = SlidingPuzzle(size=3, scramble_moves=SCRAMBLE, max_steps=50)
engine = MCTSEngine(game, iterations=ITERS, max_rollout_depth=50)

print(f'Playing {NUM_GAMES} Sliding Puzzle games '
      f'(scramble={SCRAMBLE}, iters={ITERS}, random rollouts)...')
baseline_stats = engine.play_many(
    num_games=NUM_GAMES,
    clear_table_each_game=True,
)
print(f'\nBaseline stats: {baseline_stats}')
print(f'Solve rate: {baseline_stats["win_rate"]*100:.1f}%')

Playing 30 Sliding Puzzle games (scramble=10, iters=30, random rollouts)...
  Game 3/30: unsolved, moves=50
  Game 6/30: solved, moves=44
  Game 9/30: solved, moves=40
  Game 12/30: solved, moves=24
  Game 15/30: unsolved, moves=50
  Game 18/30: unsolved, moves=50
  Game 21/30: unsolved, moves=50
  Game 24/30: solved, moves=50
  Game 27/30: solved, moves=50
  Game 30/30: unsolved, moves=50

Baseline stats: {'total_games': 30, 'wins': 12, 'losses': 0, 'draws': 18, 'win_rate': 0.4}
Solve rate: 40.0%


In [14]:
# Cell 6: Inspect traces — look at unsolved games
report = engine.logger.format_for_llm(max_games=3)
print(report[:2000])

MCTS GAMEPLAY TRACE  ({'total_games': 30, 'wins': 12, 'losses': 0, 'draws': 18, 'win_rate': 0.4})

--- Game 1 | Winner: Player None | MCTS was Player 0 | Moves: 50 | Time: 0.031s ---
  Turn 1: P0 chose action=1 (visits=32, top=[{'action': 1, 'visits': 17, 'avg_value': 0.0}, {'action': 2, 'visits': 17, 'avg_value': 0.0}])
    Board:
      Step 0/50 | Misplaced: 7 | Manhattan: 10
      +--+--+--+
      |1 |3 |  |
      +--+--+--+
      |7 |2 |4 |
      +--+--+--+
      |8 |6 |5 |
      +--+--+--+
  Turn 2: P0 chose action=0 (visits=47, top=[{'action': 0, 'visits': 32, 'avg_value': 0.0}, {'action': 1, 'visits': 24, 'avg_value': 0.0}, {'action': 2, 'visits': 23, 'avg_value': 0.0}])
    Board:
      Step 1/50 | Misplaced: 7 | Manhattan: 11
      +--+--+--+
      |1 |3 |4 |
      +--+--+--+
      |7 |2 |  |
      +--+--+--+
      |8 |6 |5 |
      +--+--+--+
  Turn 3: P0 chose action=1 (visits=62, top=[{'action': 1, 'visits': 47, 'avg_value': 0.0}, {'action': 2, 'visits': 47, 'avg_value': 0.0

## Part 3: Component 2 — LLM Proposes Better Heuristics

Using `PromptBuilder` to assemble game-specific prompts and `LLMClient` to call Qwen 3.5, then `HeuristicLoader` to safely extract+ compile the code.

In [15]:
# Cell 7: Build the LLM prompt using PromptBuilder
builder = PromptBuilder()  # auto-detects template from game name

prompt = builder.build(engine, baseline_stats, prev_code=None)

print(f'Prompt length: {len(prompt)} chars')
print('\nFirst 500 chars of prompt:')
print(prompt[:500])

Prompt length: 5107 chars

First 500 chars of prompt:
You are an expert game AI engineer optimizing MCTS heuristics for a puzzle/game.

## Game: Sliding Puzzle (8-puzzle)
- 3x3 grid with tiles 1-8 and one blank space
- Actions: slide the blank UP(0), DOWN(1), LEFT(2), RIGHT(3)
- Goal: arrange tiles as [1,2,3,4,5,6,7,8,_] (blank at bottom-right)
- The puzzle state has helper methods:
  - state.misplaced_tiles() -> int (tiles not in goal position)
  - state.manhattan_distance() -> int (sum of Manhattan distances to goals)
  - state.board -> list[int]


In [16]:
# Cell 8: Call Qwen 3.5 via LLMClient
llm = LLMClient(model='qwen3.5:35b')
print(f'LLM available: {llm.is_available()}')

resp = llm.chat(prompt)

print(f'=== LLM Response ({resp.elapsed_sec}s) ===')
print(resp.content[:1500] if resp.content.strip() else resp.thinking[:1500])

LLM available: True
=== LLM Response (54.9s) ===
```python
def evaluation(state: GameState, perspective_player: int) -> float | None:
    """
    Evaluate puzzle state using Manhattan distance heuristic.
    
    Returns float in [0, 1] where 1.0 = solved, lower = farther from goal.
    Uses strong contrast: solved=1.0, scrambled with md=10~0.09, md=20~0.05
    """
    md = state.manhattan_distance()
    if md == 0:
        return 1.0
    return 1.0 / (1.0 + md)
```


In [17]:
# Cell 9: Extract and load the improved evaluation function
loader = HeuristicLoader(game_state_classes=[SlidingPuzzleState])

test_state = SlidingPuzzle(size=3, scramble_moves=5).new_initial_state()
improved_eval, code = loader.load_from_response(
    resp.full_text,
    target_name='evaluation',
    test_state=test_state,
)

print(f'Loaded function: {improved_eval.__name__}')
print(f'\n=== Extracted Code ===')
print(code)
print(f'\nTest evaluation on scrambled state: {improved_eval(test_state, 0)}')

Loaded function: evaluation

=== Extracted Code ===
def evaluation(state: GameState, perspective_player: int) -> float | None:
    """
    Evaluate puzzle state using Manhattan distance heuristic.
    
    Returns float in [0, 1] where 1.0 = solved, lower = farther from goal.
    Uses strong contrast: solved=1.0, scrambled with md=10~0.09, md=20~0.05
    """
    md = state.manhattan_distance()
    if md == 0:
        return 1.0
    return 1.0 / (1.0 + md)

Test evaluation on scrambled state: 0.16666666666666666


## Part 4: Hot-Swap & Re-Evaluate

Replace the default evaluation with the LLM's version and re-run the same puzzles.

In [18]:
# Cell 10: Compare baseline vs LLM-improved
game2 = SlidingPuzzle(size=3, scramble_moves=SCRAMBLE, max_steps=50)
engine2 = MCTSEngine(game2, iterations=ITERS, max_rollout_depth=50)

# Hot-swap the evaluation heuristic
engine2.set_heuristic('evaluation', improved_eval)
print(f'Swapped evaluation -> {improved_eval.__name__}')

print(f'\nPlaying {NUM_GAMES} games with LLM-improved evaluation...')
improved_stats = engine2.play_many(
    num_games=NUM_GAMES,
    clear_table_each_game=True,
)

print(f'\n{"="*50}')
print(f'BASELINE  solve rate: {baseline_stats["win_rate"]*100:.1f}% '
      f'({baseline_stats["wins"]}/{baseline_stats["total_games"]})')
print(f'IMPROVED  solve rate: {improved_stats["win_rate"]*100:.1f}% '
      f'({improved_stats["wins"]}/{improved_stats["total_games"]})')
delta = (improved_stats['win_rate'] - baseline_stats['win_rate']) * 100
print(f'IMPROVEMENT: {delta:+.1f} percentage points')
print(f'{"="*50}')

Swapped evaluation -> evaluation

Playing 30 games with LLM-improved evaluation...
  Game 3/30: unsolved, moves=50
  Game 6/30: unsolved, moves=50
  Game 9/30: solved, moves=38
  Game 12/30: solved, moves=36
  Game 15/30: unsolved, moves=50
  Game 18/30: unsolved, moves=50
  Game 21/30: solved, moves=42
  Game 24/30: solved, moves=18
  Game 27/30: solved, moves=18
  Game 30/30: solved, moves=18

BASELINE  solve rate: 40.0% (12/30)
IMPROVED  solve rate: 53.3% (16/30)
IMPROVEMENT: +13.3 percentage points


## Part 5: Full Automated Multi-Round Optimization Loop

`OptimizationLoop` encapsulates the entire play→LLM→validate→adopt cycle.
It runs multiple rounds, tracks the best heuristic, and only keeps improvements.

In [19]:
# Cell 11: Automated multi-round optimization loop (encapsulated)

loop = OptimizationLoop(
    game_factory=lambda: SlidingPuzzle(size=3, scramble_moves=10, max_steps=50),
    state_classes=[SlidingPuzzleState],
    mcts_iterations=30,
    max_rollout_depth=50,
    games_per_round=30,
    validation_games=30,
    target_heuristic='evaluation',
    llm_client=LLMClient(model='qwen3.5:35b'),
)

result = loop.run(num_rounds=3, verbose=True)
print(f'\n{result.summary()}')

Measuring baseline (default heuristics)...
  Game 3/30: unsolved, moves=50
  Game 6/30: unsolved, moves=50
  Game 9/30: solved, moves=22
  Game 12/30: solved, moves=44
  Game 15/30: unsolved, moves=50
  Game 18/30: unsolved, moves=50
  Game 21/30: solved, moves=36
  Game 24/30: unsolved, moves=50
  Game 27/30: unsolved, moves=50
  Game 30/30: solved, moves=38
Baseline solve rate: 33.3%

ROUND 1/3
  Game 3/30: solved, moves=48
  Game 6/30: solved, moves=48
  Game 9/30: unsolved, moves=50
  Game 12/30: unsolved, moves=50
  Game 15/30: unsolved, moves=50
  Game 18/30: unsolved, moves=50
  Game 21/30: solved, moves=50
  Game 24/30: solved, moves=18
  Game 27/30: unsolved, moves=50
  Game 30/30: unsolved, moves=50
  Play: 50.0% solved (0.6s)
  Calling LLM (5107 char prompt)...
  LLM responded in 46.3s
  Extracted code (492 chars)
  Game 3/30: solved, moves=34
  Game 6/30: unsolved, moves=50
  Game 9/30: solved, moves=46
  Game 12/30: solved, moves=46
  Game 15/30: unsolved, moves=50
  Game 

## Part 6 (Optional): Test on Sokoban

Same optimization loop, different game.

In [20]:
# Cell 12: Sokoban baseline + (optional) optimization loop
sok_game = Sokoban(level_name='micro3', max_steps=100)
sok_engine = MCTSEngine(sok_game, iterations=30, max_rollout_depth=30)

print('Sokoban micro3 baseline (2 boxes)...')
sok_stats = sok_engine.play_many(num_games=10, clear_table_each_game=True)
print(f'Solve rate: {sok_stats["win_rate"]*100:.0f}%')
print(f'\nUnsolved trace:')
print(sok_engine.logger.format_for_llm(max_games=2)[:1500])

# To run the full optimization loop on Sokoban, uncomment below:
# sok_loop = OptimizationLoop(
#     game_factory=lambda: Sokoban(level_name='micro3', max_steps=100),
#     state_classes=[SokobanState],
#     mcts_iterations=30,
#     max_rollout_depth=30,
#     games_per_round=10,
#     validation_games=10,
# )
# sok_result = sok_loop.run(num_rounds=3, verbose=True)
# print(sok_result.summary())

Sokoban micro3 baseline (2 boxes)...
  Game 1/10: unsolved, moves=100
  Game 2/10: unsolved, moves=100
  Game 3/10: unsolved, moves=100
  Game 4/10: unsolved, moves=100
  Game 5/10: unsolved, moves=100
  Game 6/10: unsolved, moves=100
  Game 7/10: unsolved, moves=100
  Game 8/10: unsolved, moves=100
  Game 9/10: unsolved, moves=100
  Game 10/10: unsolved, moves=100
Solve rate: 0%

Unsolved trace:
MCTS GAMEPLAY TRACE  ({'total_games': 10, 'wins': 0, 'losses': 0, 'draws': 10, 'win_rate': 0.0})

--- Game 1 | Winner: Player None | MCTS was Player 0 | Moves: 100 | Time: 0.069s ---
  Turn 1: P0 chose action=3 (visits=33, top=[{'action': 3, 'visits': 12, 'avg_value': 0.0}, {'action': 0, 'visits': 12, 'avg_value': 0.0}, {'action': 2, 'visits': 11, 'avg_value': 0.0}])
    Board:
      Step 0/100 | Boxes on target: 0/2 | Total distance: 2
        ####  
      ###  ###
      # $ .  #
      # .$ @ #
      ###  ###
        ####  
  Turn 2: P0 chose action=0 (visits=42, top=[{'action': 0, 'visits': 